# Can we predict a warranty claim within a year?
Target: `claim_within_year` (1 = claim filed within a year of sale). Approach: explore, hold out a test set, compare a few models with cross-validation, then evaluate once on the held-out set and check whether the signal is useful.

In [ ]:
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve, precision_recall_curve, brier_score_loss
from sklearn.calibration import calibration_curve
from sklearn.inspection import permutation_importance
RS=42
df=pd.read_csv('data.csv')
df.shape

## 1. Data checks

In [ ]:
print(df.dtypes); print(df.isna().sum().sum(),'missing;', df.device_serial.duplicated().sum(),'duplicate serials')
df.describe().T

In [ ]:
y=df.claim_within_year
print(f'Base rate: {y.mean():.3f}  ({y.sum()} claims of {len(y)})')
df.groupby('channel').claim_within_year.agg(['mean','count'])

The positive class is ~12.6%, so accuracy is misleading (always predicting "no claim" gives ~87%). I use ROC AUC and average precision (PR AUC; the no-skill level equals the base rate). `device_serial` is an identifier, so it is excluded from the features.

In [ ]:
num=['device_age_years','price_eur','daily_hours']
fig,ax=plt.subplots(1,3,figsize=(14,3.5))
for a,c in zip(ax,num):
    q=pd.qcut(df[c],10,duplicates='drop')
    r=df.groupby(q,observed=True).claim_within_year.mean()
    a.plot(range(len(r)),r.values,'o-'); a.axhline(y.mean(),ls='--',c='gray')
    a.set_title(f'Claim rate by {c} decile'); a.set_xlabel('decile (low to high)'); a.set_ylim(0)
plt.tight_layout(); plt.show()
df[num].describe().loc[['min','50%','max']]

Claim rate rises with device age, daily use and price. Price and daily hours are right-skewed, so they are log-transformed for the linear model.

## 2. Split and model comparison

In [ ]:
X=df[['channel']+num]
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=0.25,stratify=y,random_state=RS)
print(len(Xtr),len(Xte),ytr.mean().round(3),yte.mean().round(3))
log=FunctionTransformer(np.log1p,feature_names_out='one-to-one')
lin_pre=ColumnTransformer([('c',OneHotEncoder(drop='first'),['channel']),
    ('l',Pipeline([('log',log),('s',StandardScaler())]),['price_eur','daily_hours']),
    ('n',StandardScaler(),['device_age_years'])])
tree_pre=ColumnTransformer([('c',OneHotEncoder(),['channel'])],remainder='passthrough')
models={
 'baseline (prior)':DummyClassifier(strategy='prior'),
 'logistic regression':Pipeline([('p',lin_pre),('m',LogisticRegression(max_iter=1000))]),
 'gradient boosting':Pipeline([('p',tree_pre),('m',HistGradientBoostingClassifier(max_depth=3,learning_rate=0.05,max_iter=150,l2_regularization=1.0,random_state=RS))]),
}
cv=StratifiedKFold(5,shuffle=True,random_state=RS)
rows={}
for n,m in models.items():
    s=cross_validate(m,Xtr,ytr,cv=cv,scoring={'auc':'roc_auc','ap':'average_precision','brier':'neg_brier_score'})
    rows[n]={'auc':s['test_auc'].mean(),'auc_sd':s['test_auc'].std(),'ap':s['test_ap'].mean(),'brier':-s['test_brier'].mean()}
pd.DataFrame(rows).T.round(3)

## 3. Held-out test performance

In [ ]:
lr=models['logistic regression'].fit(Xtr,ytr); gb=models['gradient boosting'].fit(Xtr,ytr)
res={}
prior_brier=brier_score_loss(yte,np.full(len(yte),ytr.mean()))
for n,m in [('logistic regression',lr),('gradient boosting',gb)]:
    p=m.predict_proba(Xte)[:,1]; res[n]=p
    print(f'{n}: AUC={roc_auc_score(yte,p):.3f}  AP={average_precision_score(yte,p):.3f} (no-skill {yte.mean():.3f})  Brier={brier_score_loss(yte,p):.4f} (prior {prior_brier:.4f})')
rng=np.random.default_rng(RS); p=res['logistic regression']; yt=yte.values
b=[]
for _ in range(1000):
    i=rng.integers(0,len(yt),len(yt)); b.append(roc_auc_score(yt[i],p[i]))
print('Logistic AUC 95% bootstrap CI:',np.percentile(b,[2.5,97.5]).round(3))
fig,ax=plt.subplots(1,3,figsize=(15,4))
for n,p in res.items():
    f,t,_=roc_curve(yte,p); ax[0].plot(f,t,label=n)
    pr,rc,_=precision_recall_curve(yte,p); ax[1].plot(rc,pr,label=n)
    fp,mp=calibration_curve(yte,p,n_bins=8,strategy='quantile'); ax[2].plot(mp,fp,'o-',label=n)
ax[0].plot([0,1],[0,1],'--',c='gray'); ax[0].set_title('ROC'); ax[0].legend()
ax[1].axhline(yte.mean(),ls='--',c='gray'); ax[1].set_title('Precision-recall'); ax[1].set_xlabel('recall')
ax[2].plot([0,.4],[0,.4],'--',c='gray'); ax[2].set_title('Calibration'); ax[2].set_xlabel('predicted'); ax[2].set_ylabel('observed')
plt.tight_layout(); plt.show()

### What does the signal buy us in practice? (lift by risk quintile)

In [ ]:
p=res['logistic regression']
t=pd.DataFrame({'p':p,'y':yte.values}); t['bucket']=pd.qcut(t.p.rank(method='first'),5,labels=['lowest 20%','2','3','4','highest 20%'])
g=t.groupby('bucket',observed=True).agg(n=('y','size'),claim_rate=('y','mean'),mean_pred=('p','mean'))
g['lift']=g.claim_rate/yte.mean(); g.round(3)

### Drivers

In [ ]:
pi=permutation_importance(lr,Xte,yte,scoring='roc_auc',n_repeats=20,random_state=RS)
print(pd.Series(pi.importances_mean,index=X.columns).sort_values(ascending=False).round(4))
print(pd.Series(lr[-1].coef_[0],index=lr[:-1].get_feature_names_out()).round(3))

## 4. Sanity check: the identifier carries no signal
`device_serial` should be noise. Signal here would suggest leakage or an ordering artefact.

In [ ]:
s=cross_validate(HistGradientBoostingClassifier(random_state=RS),df[['device_serial']],y,cv=cv,scoring='roc_auc')
print('CV AUC from device_serial alone:',s['test_score'].mean().round(3))